# Diagnóstico de consumo de memória — Dashboard ENEM

Notebook para investigar o consumo de memória das quatro bases Parquet utilizadas por `read_allfiles2()` e separar:

- custo de importação do ambiente;
- custo de leitura individual de cada Parquet;
- efeito da projeção de colunas e dos filtros territoriais;
- memória profunda dos DataFrames;
- custo da harmonização territorial;
- memória liberada após descarte;
- custo do carregamento integrado.

**Importante:** execute primeiro as células de configuração e inventário. Os testes isolados são executados em processos filhos para reduzir a interferência entre medições. O notebook não altera os Parquets nem o código do dashboard.

## 1. Configuração e ambiente

Execute a partir da raiz de `app_render` ou ajuste `PROJECT_ROOT` manualmente. O notebook importa os caminhos de `dash_utils.config`, evitando duplicar a configuração do projeto.

In [1]:
from pathlib import Path
import os
import sys
import json
import gc
import time
import subprocess
import textwrap
import importlib
import pandas as pd
import numpy as np
import psutil
import pyarrow.parquet as pq
from IPython.display import display

# Ajuste se o notebook não estiver dentro de app_render.
PROJECT_ROOT = Path.cwd()

# Procura a raiz que contém dash_utils/config.py.
if not (PROJECT_ROOT / "dash_utils" / "config.py").exists():
    candidatos = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / "app_render",
        Path.home() / "PycharmProjects" / "Projeto_ENEM" / "app_render",
    ]
    PROJECT_ROOT = next(
        (p.resolve() for p in candidatos if (p / "dash_utils" / "config.py").exists()),
        None
    )

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Não encontrei dash_utils/config.py. Defina PROJECT_ROOT manualmente."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

config = importlib.import_module("dash_utils.config")
CONSUMO_DIR = Path(config.CONSUMO_DIR)
DATA_DIR = Path(config.DATA_DIR)

UF = "PA"
REFERENCIA = "ESC"   # opções: "ESC", "PROVA" ou None

UF_DIR = CONSUMO_DIR / f"uf={UF}"
ARQUIVOS = {
    "painel": UF_DIR / "painel_municipal.parquet",
    "faixa": UF_DIR / "faixa_etaria_municipal.parquet",
    "grupo": UF_DIR / "desempenho_grupos_municipal.parquet",
    "dsp": UF_DIR / "desempenho_sexo_faixa_etaria_municipal.parquet",
}
SHAPEFILE = DATA_DIR / "shapefiles" / f"{UF}_Municipios_2025" / f"{UF}_Municipios_2025.shp"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONSUMO_DIR:", CONSUMO_DIR)
print("DATA_DIR:", DATA_DIR)
print("UF:", UF, "| referência:", REFERENCIA)
print("Processo:", os.getpid())
print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("PyArrow:", importlib.import_module("pyarrow").__version__)
print("psutil:", psutil.__version__)

PROJECT_ROOT: /home/akel/PycharmProjects/Projeto_ENEM/app_render
CONSUMO_DIR: /home/akel/PycharmProjects/Projeto_ENEM/app_render/data/consumo
DATA_DIR: /home/akel/PycharmProjects/Projeto_ENEM/app_render/data
UF: PA | referência: ESC
Processo: 27763
Python: 3.11.18 (main, Oct 26 2026, 11:35:17) [GCC 15.3.0]
Pandas: 3.1.0
PyArrow: 26.0.1
psutil: 7.2.2


## 2. Funções de medição

RSS representa a memória residente total do processo; USS estima a memória exclusiva do processo. A memória profunda do pandas é uma medida do conteúdo dos DataFrames e **não deve ser somada diretamente ao RSS**, pois os conceitos se sobrepõem.

In [2]:
PROCESS = psutil.Process()

def memoria_processo():
    info = PROCESS.memory_full_info()
    return {
        "rss_mb": info.rss / 1024**2,
        "uss_mb": getattr(info, "uss", np.nan) / 1024**2,
    }

def memoria_df(df):
    return float(df.memory_usage(index=True, deep=True).sum() / 1024**2)

def bytes_mb(valor):
    return valor / 1024**2

display(pd.DataFrame([memoria_processo()]))

,rss_mb,uss_mb
0,138.847656,119.152344


## 3. Inventário dos arquivos Parquet

Verifica tamanho em disco, linhas, row groups, colunas físicas e esquema. Esta etapa lê metadados, não carrega os dados completos.

In [3]:
inventario = []

for nome, caminho in ARQUIVOS.items():
    if not caminho.exists():
        inventario.append({"base": nome, "arquivo": str(caminho), "erro": "arquivo não encontrado"})
        continue

    pf = pq.ParquetFile(caminho)
    meta = pf.metadata
    inventario.append({
        "base": nome,
        "arquivo": caminho.name,
        "tamanho_mb": bytes_mb(caminho.stat().st_size),
        "linhas": meta.num_rows,
        "row_groups": meta.num_row_groups,
        "colunas": len(pf.schema_arrow.names),
        "row_group_mb_medio_descompactado": (
            np.mean([meta.row_group(i).total_byte_size for i in range(meta.num_row_groups)]) / 1024**2
            if meta.num_row_groups else np.nan
        ),
    })

df_inventario = pd.DataFrame(inventario)
display(df_inventario)

,base,arquivo,tamanho_mb,linhas,row_groups,colunas,row_group_mb_medio_descompactado
0,painel,painel_municipal.parquet,1.185326,6464,1,82,1.645310
1,faixa,faixa_etaria_municipal.parquet,0.144585,63836,1,7,0.199375
2,grupo,desempenho_grupos_municipal.parquet,8.573952,337214,1,16,10.338845
3,dsp,desempenho_sexo_faixa_etaria_municipal.parquet,10.102821,451205,1,17,12.507343


In [4]:
# Esquema completo de cada arquivo: nome, tipo e nulabilidade.
for nome, caminho in ARQUIVOS.items():
    if caminho.exists():
        print(f"\n### {nome}: {caminho.name}")
        print(pq.ParquetFile(caminho).schema_arrow)


### painel: painel_municipal.parquet
ano: int64
CO_MUNICIPIO_TERRITORIO: large_string
NO_MUNICIPIO_TERRITORIO: large_string
TIPO_REFERENCIA_TERRITORIAL: large_string
n_participantes: int64
n_masculino: int64
n_feminino: int64
n_faixa_etaria_informada: int64
pct_masculino: double
pct_feminino: double
n_registros_presenca: int64
n_presentes: int64
n_ausentes: int64
n_presenca_informada: int64
pct_presenca: double
pct_ausencia: double
n_validos_redacao: int64
n_zeros_redacao: int64
media_redacao: double
media_positivas_redacao: double
mediana_redacao: double
std_redacao: double
q1_redacao: double
q3_redacao: double
n_avaliados_redacao: int64
n_validos_objetiva: double
n_zeros_objetiva: double
media_objetiva: double
media_positivas_objetiva: double
mediana_objetiva: double
std_objetiva: double
q1_objetiva: double
q3_objetiva: double
n_avaliados_objetiva: double
n_validos_cn: double
n_zeros_cn: double
media_cn: double
media_positivas_cn: double
mediana_cn: double
std_cn: double
q1_cn: doub

## 4. Projeções efetivamente usadas pelo dashboard

As listas abaixo reproduzem as projeções existentes no `features_names.py` enviado. Elas serão usadas como referência nos testes de leitura.

In [5]:
COLUNAS = {
    "painel": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "n_participantes", "n_masculino", "n_feminino",
        "n_faixa_etaria_informada", "n_escola_federal",
        "n_escola_estadual", "n_escola_municipal", "n_escola_privada",
        "n_escola_nao_informada", "media_redacao", "n_validos_redacao",
        "media_objetiva", "n_validos_objetiva", "media_cn", "n_validos_cn",
        "media_ch", "n_validos_ch", "media_lc", "n_validos_lc",
        "media_mt", "n_validos_mt",
    ],
    "faixa": [
        "ano", "CO_MUNICIPIO_TERRITORIO",
        "TIPO_REFERENCIA_TERRITORIAL", "TP_FAIXA_ETARIA", "n_participantes",
    ],
    "grupo": [
        "ano", "CO_MUNICIPIO_TERRITORIO",
        "TIPO_REFERENCIA_TERRITORIAL", "tipo_grupo", "grupo",
        "disciplina", "media", "n_validos", "mediana", "std", "q1", "q3",
    ],
    "dsp": [
        "ano", "CO_MUNICIPIO_TERRITORIO",
        "TIPO_REFERENCIA_TERRITORIAL", "sexo", "faixa_etaria",
        "disciplina", "n_participantes", "n_validos", "media", "std",
    ],
}

for nome, cols in COLUNAS.items():
    print(f"{nome}: {len(cols)} colunas projetadas")

painel: 24 colunas projetadas
faixa: 5 colunas projetadas
grupo: 12 colunas projetadas
dsp: 10 colunas projetadas


## 5. Leitura isolada em processo filho

Cada cenário roda em um novo processo Python. O relatório separa:

- memória após importar bibliotecas;
- memória imediatamente após a leitura;
- memória profunda do DataFrame;
- memória após excluir o DataFrame e executar `gc.collect()`.

Isso evita que uma leitura anterior ou um cache do notebook contamine a medição seguinte. O delta RSS/USS é calculado em relação ao processo já inicializado com as bibliotecas.

In [6]:
WORKER_CODE = r'''
import os, sys, json, gc, time
from pathlib import Path
import pandas as pd
import psutil

payload = json.loads(sys.argv[1])
proc = psutil.Process()

def mem():
    x = proc.memory_full_info()
    return {"rss_mb": x.rss / 1024**2, "uss_mb": getattr(x, "uss", 0) / 1024**2}

resultado = {"base": payload["base"], "cenario": payload["cenario"]}
m0 = mem()
t0 = time.perf_counter()

df = pd.read_parquet(
    payload["arquivo"],
    engine="pyarrow",
    columns=payload.get("columns"),
    filters=payload.get("filters"),
)

t1 = time.perf_counter()
m1 = mem()
resultado.update({
    "linhas_lidas": len(df),
    "colunas_lidas": len(df.columns),
    "tempo_leitura_s": t1 - t0,
    "rss_antes_mb": m0["rss_mb"],
    "uss_antes_mb": m0["uss_mb"],
    "rss_depois_leitura_mb": m1["rss_mb"],
    "uss_depois_leitura_mb": m1["uss_mb"],
    "delta_rss_leitura_mb": m1["rss_mb"] - m0["rss_mb"],
    "delta_uss_leitura_mb": m1["uss_mb"] - m0["uss_mb"],
    "dataframe_deep_mb": float(df.memory_usage(index=True, deep=True).sum() / 1024**2),
    "dtypes": {str(k): str(v) for k, v in df.dtypes.items()},
})

del df
gc.collect()
m2 = mem()
resultado.update({
    "rss_apos_del_mb": m2["rss_mb"],
    "uss_apos_del_mb": m2["uss_mb"],
    "delta_uss_retido_mb": m2["uss_mb"] - m0["uss_mb"],
})

print(json.dumps(resultado))
'''

def executar_leitura_isolada(base, columns=None, filters=None, cenario="projecao_dashboard"):
    payload = {
        "base": base,
        "cenario": cenario,
        "arquivo": str(ARQUIVOS[base]),
        "columns": columns,
        "filters": filters,
    }
    proc = subprocess.run(
        [sys.executable, "-c", WORKER_CODE, json.dumps(payload)],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"Falha no teste {base}/{cenario}\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    linhas = [x for x in proc.stdout.splitlines() if x.strip().startswith("{")]
    if not linhas:
        raise RuntimeError(f"Worker não retornou JSON: {proc.stdout}")
    return json.loads(linhas[-1])

## 6. Teste principal: quatro bases com projeção do dashboard

Executa cada base separadamente, sem filtro territorial, para comparar o custo bruto da projeção usada pelo dashboard. O processo filho reinicia a cada base.

In [7]:
resultados_projecao = []

for base in ARQUIVOS:
    resultado = executar_leitura_isolada(
        base,
        columns=COLUNAS[base],
        filters=None,
        cenario="projecao_dashboard_sem_filtro",
    )
    resultados_projecao.append(resultado)
    print(
        base,
        "| linhas:", resultado["linhas_lidas"],
        "| delta USS:", round(resultado["delta_uss_leitura_mb"], 1), "MB",
        "| DataFrame:", round(resultado["dataframe_deep_mb"], 1), "MB",
        "| retido após del:", round(resultado["delta_uss_retido_mb"], 1), "MB",
    )

df_resultados_projecao = pd.DataFrame(resultados_projecao)
display(df_resultados_projecao[[
    "base", "linhas_lidas", "colunas_lidas", "tempo_leitura_s",
    "delta_rss_leitura_mb", "delta_uss_leitura_mb",
    "dataframe_deep_mb", "delta_uss_retido_mb",
]])

painel | linhas: 6464 | delta USS: 38.5 MB | DataFrame: 1.3 MB | retido após del: 38.8 MB
faixa | linhas: 63836 | delta USS: 49.7 MB | DataFrame: 3.2 MB | retido após del: 50.0 MB
grupo | linhas: 337214 | delta USS: 159.0 MB | DataFrame: 40.2 MB | retido após del: 156.7 MB
dsp | linhas: 451205 | delta USS: 149.9 MB | DataFrame: 45.3 MB | retido após del: 146.8 MB


,base,linhas_lidas,colunas_lidas,tempo_leitura_s,delta_rss_leitura_mb,delta_uss_leitura_mb,dataframe_deep_mb,delta_uss_retido_mb
0,painel,6464,24,0.026214,45.886719,38.507812,1.298841,38.832031
1,faixa,63836,5,0.021337,57.144531,49.714844,3.244058,50.035156
2,grupo,337214,12,0.046415,166.398438,158.976562,40.174284,156.726562
3,dsp,451205,10,0.049007,157.253906,149.898438,45.293781,146.777344


## 7. Efeito dos filtros territoriais

O `features_names.py` usa filtro Arrow na leitura (`ESC` inclui `ESC` e `RESIDENCIA`; `PROVA` inclui `PROVA`) e depois normaliza `RESIDENCIA` para `ESC` e aplica novamente o filtro no pandas.

Aqui comparamos três cenários por base:
- sem filtro;
- filtro ESC;
- filtro PROVA.

A comparação permite avaliar quanto o filtro pushdown reduz linhas e memória de leitura. Se o Parquet não estiver particionado/organizado de modo favorável, a redução de linhas pode não produzir redução proporcional de RSS.

In [8]:
def filtros_referencia(ref):
    if ref == "ESC":
        valores = ["ESC", "RESIDENCIA"]
    elif ref == "PROVA":
        valores = ["PROVA"]
    else:
        return None
    return [("TIPO_REFERENCIA_TERRITORIAL", "in", valores)]

resultados_filtros = []
for base in ARQUIVOS:
    for ref in [None, "ESC", "PROVA"]:
        r = executar_leitura_isolada(
            base,
            columns=COLUNAS[base],
            filters=filtros_referencia(ref),
            cenario=f"referencia_{ref or 'todas'}",
        )
        r["referencia"] = ref or "TODAS"
        resultados_filtros.append(r)

df_filtros = pd.DataFrame(resultados_filtros)
display(df_filtros[[
    "base", "referencia", "linhas_lidas", "tempo_leitura_s",
    "delta_rss_leitura_mb", "delta_uss_leitura_mb", "dataframe_deep_mb",
]])

,base,referencia,linhas_lidas,tempo_leitura_s,delta_rss_leitura_mb,delta_uss_leitura_mb,dataframe_deep_mb
0,painel,TODAS,6464,0.026123,47.691406,40.019531,1.298841
1,painel,ESC,5125,0.028473,46.746094,39.042969,1.028575
2,painel,PROVA,1339,0.024894,42.511719,34.753906,0.271164
3,faixa,TODAS,63836,0.021303,57.070312,49.394531,3.244058
4,faixa,ESC,40659,0.022799,60.046875,52.339844,2.046778
5,faixa,PROVA,23177,0.025870,67.753906,59.976562,1.212629
6,grupo,TODAS,337214,0.045944,170.234375,162.531250,40.174284
7,grupo,ESC,205400,0.054353,186.078125,178.359375,24.440821
8,grupo,PROVA,131814,0.051776,173.253906,165.527344,15.894396
9,dsp,TODAS,451205,0.050424,163.351562,155.648438,45.293781


## 8. Custo da projeção de colunas

Compara, em processos independentes, a projeção do dashboard com a leitura de todas as colunas. O filtro territorial permanece desativado para isolar o efeito das colunas.

In [9]:
resultados_colunas = []

for base, caminho in ARQUIVOS.items():
    # Projeção real do dashboard
    resultados_colunas.append(
        executar_leitura_isolada(
            base, columns=COLUNAS[base], filters=None,
            cenario="projecao_dashboard"
        )
    )
    # Todas as colunas
    resultados_colunas.append(
        executar_leitura_isolada(
            base, columns=None, filters=None,
            cenario="todas_as_colunas"
        )
    )

df_colunas = pd.DataFrame(resultados_colunas)
display(df_colunas[[
    "base", "cenario", "linhas_lidas", "colunas_lidas",
    "delta_uss_leitura_mb", "dataframe_deep_mb", "tempo_leitura_s",
]])

,base,cenario,linhas_lidas,colunas_lidas,delta_uss_leitura_mb,dataframe_deep_mb,tempo_leitura_s
0,painel,projecao_dashboard,6464,24,40.015625,1.298841,0.026960
1,painel,todas_as_colunas,6464,82,56.695312,4.280939,0.037577
2,faixa,projecao_dashboard,63836,5,47.371094,3.244058,0.021828
3,faixa,todas_as_colunas,63836,7,67.441406,5.183252,0.026835
4,grupo,projecao_dashboard,337214,12,160.585938,40.174284,0.048934
5,grupo,todas_as_colunas,337214,16,198.421875,55.466128,0.059999
6,dsp,projecao_dashboard,451205,10,149.671875,45.293781,0.050903
7,dsp,todas_as_colunas,451205,17,227.648438,76.008500,0.067333


## 9. Inspeção dos tipos e memória profunda

Analisa a memória profunda de cada coluna para identificar campos `object` potencialmente caros, strings repetidas e colunas numéricas com tipos maiores do que o necessário. Esta etapa usa uma leitura no processo do notebook; faça-a depois dos testes isolados.

In [10]:
def relatorio_colunas(df, nome):
    memoria = df.memory_usage(index=False, deep=True).sort_values(ascending=False)
    tabela = pd.DataFrame({
        "coluna": memoria.index,
        "memoria_mb": memoria.values / 1024**2,
        "dtype": [str(df[c].dtype) for c in memoria.index],
        "n_unicos": [df[c].nunique(dropna=False) for c in memoria.index],
        "proporcao_unicos": [
            df[c].nunique(dropna=False) / max(len(df), 1) for c in memoria.index
        ],
    })
    tabela.insert(0, "base", nome)
    return tabela

dfs_inspecao = {}
relatorios_tipos = []

for base, caminho in ARQUIVOS.items():
    dfs_inspecao[base] = pd.read_parquet(
        caminho, engine="pyarrow", columns=COLUNAS[base]
    )
    relatorios_tipos.append(relatorio_colunas(dfs_inspecao[base], base))

df_tipos = pd.concat(relatorios_tipos, ignore_index=True)
display(df_tipos.sort_values(["base", "memoria_mb"], ascending=[True, False]))

,base,coluna,memoria_mb,dtype,n_unicos,proporcao_unicos
41,dsp,sexo,7.091239,str,2,0.000004
42,dsp,CO_MUNICIPIO_TERRITORIO,6.487780,string,737,0.001633
43,dsp,TIPO_REFERENCIA_TERRITORIAL,5.090850,str,2,0.000004
44,dsp,disciplina,4.954343,str,6,0.000013
45,dsp,faixa_etaria,4.027037,str,20,0.000044
46,dsp,ano,3.872724,Int64,23,0.000051
47,dsp,n_participantes,3.442421,int64,1980,0.004388
48,dsp,n_validos,3.442421,int64,2846,0.006308
49,dsp,media,3.442421,float64,189365,0.419687
50,dsp,std,3.442421,float64,285089,0.631839


## 10. Simulação de conversões e categorias

Mede, sem modificar os DataFrames originais, o efeito potencial de converter colunas textuais repetitivas para `category` e códigos municipais para `string`. Não aplicar conversões no dashboard apenas com base neste teste: validar antes os usos e a semântica dos campos.

In [11]:
resultados_tipos = []

for base, df in dfs_inspecao.items():
    original = memoria_df(df)
    candidato = df.copy()

    colunas_texto = candidato.select_dtypes(include=["object", "string"]).columns
    convertidas = []
    for col in colunas_texto:
        # Categoriza apenas quando há repetição relevante.
        razao = candidato[col].nunique(dropna=False) / max(len(candidato), 1)
        if razao < 0.5:
            candidato[col] = candidato[col].astype("category")
            convertidas.append(col)

    resultados_tipos.append({
        "base": base,
        "memoria_original_mb": original,
        "memoria_simulada_mb": memoria_df(candidato),
        "economia_mb": original - memoria_df(candidato),
        "colunas_categorizadas": ", ".join(convertidas),
    })
    del candidato

display(pd.DataFrame(resultados_tipos))

,base,memoria_original_mb,memoria_simulada_mb,economia_mb,colunas_categorizadas
0,painel,1.298841,1.164463,0.134377,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI..."
1,faixa,3.244058,1.289422,1.954636,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI..."
2,grupo,40.174284,20.270588,19.903696,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI..."
3,dsp,45.293781,20.235275,25.058506,"CO_MUNICIPIO_TERRITORIO, TIPO_REFERENCIA_TERRI..."


## 11. Custo da preparação territorial

A implementação real:
1. substitui `RESIDENCIA` por `ESC`;
2. aplica o filtro territorial em pandas;
3. converte o código municipal para texto e mapeia o nome;
4. remove municípios sem correspondência e executa `reset_index`.

A célula abaixo mede essas operações por base, separadamente da leitura. Usa o shapefile e as funções reais de `dash_utils.features_names`.

In [12]:
features_names = importlib.import_module("dash_utils.features_names")

shape = __import__("geopandas").read_file(SHAPEFILE)
mapa_nomes = features_names._criar_mapa_nomes(shape)

def medir_preparacao(base, referencia=REFERENCIA):
    df = pd.read_parquet(
        ARQUIVOS[base],
        engine="pyarrow",
        columns=COLUNAS[base],
        filters=filtros_referencia(referencia),
    )
    antes = memoria_processo()
    deep_antes = memoria_df(df)
    t0 = time.perf_counter()

    preparado = features_names._preparar_tabela(df, mapa_nomes, referencia)

    t1 = time.perf_counter()
    depois = memoria_processo()
    return {
        "base": base,
        "linhas_antes": len(df),
        "linhas_depois": len(preparado),
        "deep_antes_mb": deep_antes,
        "deep_depois_mb": memoria_df(preparado),
        "delta_rss_preparacao_mb": depois["rss_mb"] - antes["rss_mb"],
        "delta_uss_preparacao_mb": depois["uss_mb"] - antes["uss_mb"],
        "tempo_preparacao_s": t1 - t0,
    }, df, preparado

resultados_preparacao = []
for base in ARQUIVOS:
    r, bruto, preparado = medir_preparacao(base)
    resultados_preparacao.append(r)
    print(r)
    del bruto, preparado
    gc.collect()

display(pd.DataFrame(resultados_preparacao))

{'base': 'painel', 'linhas_antes': 5125, 'linhas_depois': 3625, 'deep_antes_mb': 1.0285749435424805, 'deep_depois_mb': 0.7953338623046875, 'delta_rss_preparacao_mb': 0.65625, 'delta_uss_preparacao_mb': 0.65625, 'tempo_preparacao_s': 0.0038336890002028667}
{'base': 'faixa', 'linhas_antes': 40659, 'linhas_depois': 38335, 'deep_antes_mb': 2.0467777252197266, 'deep_depois_mb': 2.6385440826416016, 'delta_rss_preparacao_mb': 6.87890625, 'delta_uss_preparacao_mb': 6.87890625, 'tempo_preparacao_s': 0.01643336699999054}
{'base': 'grupo', 'linhas_antes': 205400, 'linhas_depois': 186449, 'deep_antes_mb': 24.440820693969727, 'deep_depois_mb': 25.655953407287598, 'delta_rss_preparacao_mb': 16.43359375, 'delta_uss_preparacao_mb': 16.43359375, 'tempo_preparacao_s': 0.04700521199993091}
{'base': 'dsp', 'linhas_antes': 263761, 'linhas_depois': 250436, 'deep_antes_mb': 26.438347816467285, 'deep_depois_mb': 29.763463020324707, 'delta_rss_preparacao_mb': 3.96875, 'delta_uss_preparacao_mb': 3.96875, 'tempo

,base,linhas_antes,linhas_depois,deep_antes_mb,deep_depois_mb,delta_rss_preparacao_mb,delta_uss_preparacao_mb,tempo_preparacao_s
0,painel,5125,3625,1.028575,0.795334,0.656250,0.656250,0.003834
1,faixa,40659,38335,2.046778,2.638544,6.878906,6.878906,0.016433
2,grupo,205400,186449,24.440821,25.655953,16.433594,16.433594,0.047005
3,dsp,263761,250436,26.438348,29.763463,3.968750,3.968750,0.058706


## 12. Teste de liberação de memória no processo do notebook

A memória RSS pode não retornar ao nível inicial mesmo após `del` e `gc.collect()`, pois o alocador pode manter arenas para reutilização. USS e a memória profunda dos objetos ajudam a interpretar o resultado. Esta célula mede o comportamento do kernel atual; para comparação rigorosa, reinicie o kernel e execute novamente.

In [13]:
gc.collect()
mem_inicio = memoria_processo()

df_teste = pd.read_parquet(
    ARQUIVOS["grupo"],
    engine="pyarrow",
    columns=COLUNAS["grupo"],
    filters=filtros_referencia(REFERENCIA),
)
mem_leitura = memoria_processo()
deep_teste = memoria_df(df_teste)

del df_teste
gc.collect()
mem_final = memoria_processo()

pd.DataFrame([
    {"etapa": "antes", **mem_inicio},
    {"etapa": "após leitura grupo", **mem_leitura},
    {"etapa": "após del + gc", **mem_final},
]).assign(dataframe_deep_mb=[0, deep_teste, 0])

,etapa,rss_mb,uss_mb,dataframe_deep_mb
0,antes,570.746094,550.527344,0.000000
1,após leitura grupo,577.886719,557.667969,24.440821
2,após del + gc,577.886719,557.667969,0.000000


## 13. Carregamento integrado pela função original

Por fim, executa `read_allfiles2()` exatamente como implementada no projeto. Essa medição inclui as quatro bases, shapefile, mapa de nomes e harmonização territorial. Compare com a soma/combinação dos testes isolados, lembrando que o processo do notebook já contém bibliotecas importadas e alguns objetos de inspeção.

In [14]:
# Limpa os objetos de inspeção antes do teste integrado.
for nome in ["dfs_inspecao", "shape", "mapa_nomes"]:
    if nome in globals():
        del globals()[nome]
gc.collect()

read_allfiles2 = features_names.read_allfiles2
mem_antes_integrado = memoria_processo()
t0 = time.perf_counter()

dados_integrados = read_allfiles2(UF, REFERENCIA)

t1 = time.perf_counter()
mem_depois_integrado = memoria_processo()

resumo_integrado = {
    "tempo_total_s": t1 - t0,
    "rss_antes_mb": mem_antes_integrado["rss_mb"],
    "uss_antes_mb": mem_antes_integrado["uss_mb"],
    "rss_depois_mb": mem_depois_integrado["rss_mb"],
    "uss_depois_mb": mem_depois_integrado["uss_mb"],
    "delta_rss_mb": mem_depois_integrado["rss_mb"] - mem_antes_integrado["rss_mb"],
    "delta_uss_mb": mem_depois_integrado["uss_mb"] - mem_antes_integrado["uss_mb"],
}
display(pd.DataFrame([resumo_integrado]))

linhas_integradas = []
for chave in ["painel", "faixa", "grupo", "dsp"]:
    df = dados_integrados[chave]["data"]
    linhas_integradas.append({
        "base": chave,
        "linhas": len(df),
        "colunas": len(df.columns),
        "dataframe_deep_mb": memoria_df(df),
        "dtypes_object": int((df.dtypes == "object").sum()),
    })
display(pd.DataFrame(linhas_integradas))

,tempo_total_s,rss_antes_mb,uss_antes_mb,rss_depois_mb,uss_depois_mb,delta_rss_mb,delta_uss_mb
0,0.27232,577.085938,556.867188,596.839844,576.621094,19.753906,19.753906


,base,linhas,colunas,dataframe_deep_mb,dtypes_object
0,painel,3625,25,0.795334,0
1,faixa,38335,6,2.638544,0
2,grupo,186449,13,25.655953,0
3,dsp,250436,11,29.763463,0


## 14. Ranking e síntese automática

Ordena as bases pelo delta USS de leitura isolada e compara com a memória profunda. O ranking é uma indicação de prioridade para investigação, não uma prova isolada da causa.

In [15]:
ranking = df_resultados_projecao[[
    "base", "linhas_lidas", "colunas_lidas", "delta_uss_leitura_mb",
    "dataframe_deep_mb", "tempo_leitura_s", "delta_uss_retido_mb",
]].sort_values("delta_uss_leitura_mb", ascending=False).reset_index(drop=True)

display(ranking)

if len(ranking):
    maior = ranking.iloc[0]
    print(
        f"Maior custo isolado: {maior['base']} — "
        f"{maior['delta_uss_leitura_mb']:.1f} MB de delta USS; "
        f"DataFrame profundo: {maior['dataframe_deep_mb']:.1f} MB."
    )

,base,linhas_lidas,colunas_lidas,delta_uss_leitura_mb,dataframe_deep_mb,tempo_leitura_s,delta_uss_retido_mb
0,grupo,337214,12,158.976562,40.174284,0.046415,156.726562
1,dsp,451205,10,149.898438,45.293781,0.049007,146.777344
2,faixa,63836,5,49.714844,3.244058,0.021337,50.035156
3,painel,6464,24,38.507812,1.298841,0.026214,38.832031


Maior custo isolado: grupo — 159.0 MB de delta USS; DataFrame profundo: 40.2 MB.


## 15. Critérios para decidir a próxima otimização

Use os resultados para classificar o problema:

- **Delta USS alto e DataFrame deep alto:** o próprio conteúdo carregado é grande; revisar colunas, tipos e filtros.
- **Delta USS alto e DataFrame deep baixo:** investigar buffers temporários, conversões internas, descompressão e cópias durante a leitura.
- **Memória após `del` permanece alta:** pode haver retenção por referências, cache ou comportamento do alocador; confirmar em processo novo.
- **Leitura isolada baixa, integrada alta:** investigar sobreposição de objetos, harmonização, shapefile, cache e coexistência das quatro bases.
- **Filtro reduz linhas, mas pouco a memória:** verificar row groups/estatísticas e organização física do Parquet; o filtro pode não evitar a leitura de muitos blocos.

### Cuidados de interpretação

1. Não some RSS de diferentes etapas: RSS é uma fotografia do processo e inclui bibliotecas e alocações anteriores.
2. Não some `DataFrame deep` ao RSS.
3. Compare preferencialmente USS e deltas obtidos em processos novos.
4. Repita cenários relevantes pelo menos duas vezes se houver grande variação.
5. Execute `ESC` e `PROVA` em sessões limpas, alterando `REFERENCIA` no início.
6. Este notebook não modifica os dados nem recomenda conversões automaticamente.